
# 03 — Register Table

**Purpose:** Publish the consumption Delta dataset as a **Unity Catalog table** for SQL users and for `analysis/queries.sql`.

| Item | Value |
|------|--------|
| Source | Delta on consumption Volume |
| Target | `nyc_taxi.yellow_trips_consumption` |
| Approach (Free Edition) | `read` + `saveAsTable` (managed table) |

---

## Setup — imports and variables

`consumption_path` must match notebook **02** output.  
`full_table` is the name SQL consumers will query (`schema.table` after `setCurrentCatalog`).

---

In [0]:
# --- Imports ---
from pyspark.sql.functions import col, countDistinct, max as spark_max, min as spark_min, sum as spark_sum, when

# --- Config ---
CATALOG = "workspace"
SCHEMA = "nyc_taxi"
TABLE_NAME = "yellow_trips_consumption"
consumption_path = f"/Volumes/{CATALOG}/{SCHEMA}/consumption/yellow_taxi"
full_table = f"{SCHEMA}.{TABLE_NAME}"

spark.catalog.setCurrentCatalog(CATALOG)

## Verify consumption Delta exists

Lists files under the consumption path. Expect `_delta_log/` and Parquet part files.  
If empty, run notebook **02** first.

---

In [0]:
# --- Ensure Delta exists on consumption Volume ---
display(dbutils.fs.ls(consumption_path))

## Register managed table (`saveAsTable`)

**Free Edition approach:**

1. `spark.catalog.dropTable(..., ignoreIfNotExists=True)` — avoids schema merge errors on re-runs
2. Read Delta from the **Volume** (physical consumption layer)
3. `saveAsTable` — copies data to **UC-managed storage** and registers metadata

> **Note:** This creates a **second copy** of the data (Volume + managed table).  
> SQL queries use the **table name**; the Volume remains the curated file layer.

**Difference from Enterprise:** production typically uses `CREATE TABLE ... LOCATION 's3://...'` with **no copy** — see illustrative block at the end.

---

In [0]:
# --- Drop old table (avoids schema merge on saveAsTable) ---
spark.catalog.setCurrentCatalog(CATALOG)
spark.catalog.dropTable(full_table, ignoreIfNotExists=True)

df = spark.read.format("delta").load(consumption_path)

(
    df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(full_table)
)

## Table documentation

Adds a table comment visible in Catalog Explorer for reviewers and analysts.``

---

In [0]:
spark.sql(f"""
COMMENT ON TABLE {full_table} IS
'Yellow taxi NYC, Jan-May 2023. Consumption layer (5 columns). Source: TLC trip record data.'
""")

## Validation — schema and properties

- `printSchema()` — column names and types (expect 5 columns)
- `describe()` — summary statistics per column
- `spark.catalog.getTable()` — table metadata (managed location, etc.)

---

In [0]:
# --- Validation: schema and table properties ---
df_registered = spark.table(full_table)
df_registered.printSchema()
display(df_registered.describe())
display(spark.catalog.getTable(full_table))

## Validation — row count and sample

Confirms the table is queryable and returns sensible data.

---

In [0]:
# --- Validation: row count and sample ---
df_registered = spark.table(full_table)
print(f"total_rows: {df_registered.count():,}")
display(df_registered.limit(10))

## Validation — date range and business sanity

Checks min/max pickup datetime and basic quality flags (e.g. no negative amounts).

---

In [0]:
# --- Validation: date range and business sanity ---
display(
    spark.table(full_table).agg(
        spark_min(col("tpep_pickup_datetime")).alias("min_pickup"),
        spark_max(col("tpep_pickup_datetime")).alias("max_pickup"),
        countDistinct(col("VendorID")).alias("vendors"),
        spark_sum(when(col("total_amount") < 0, 1).otherwise(0)).alias("neg_amount"),
    )
)

## Table ready for consumption

Run queries in **`analysis/queries.sql`** or SQL Editor:

1. Monthly average `total_amount`
2. Average `passenger_count` by hour in May 2023

---

## (Illustrative only) Enterprise — external table with `LOCATION`

Not runnable on Free Edition as-is. When consumption Delta lives on **cloud storage** with an **external location** registered in Unity Catalog:

```python
# consumption_path = "s3://my-company-datalake/curated/nyc_taxi/yellow_taxi"
#
# spark.sql(f"USE CATALOG {CATALOG}")
# spark.sql(f'''
# CREATE OR REPLACE TABLE {full_table}
# USING DELTA
# LOCATION '{consumption_path}'
# ''')
```

| Topic | Free Edition | Enterprise |
|-------|--------------|------------|
| Consumption storage | `/Volumes/.../consumption/` | `s3://` / `abfss://` / `gs://` |
| Table registration | `saveAsTable` (copy) | `LOCATION` (reference only) |
| `LOCATION '/Volumes/...'` | **Not supported** (`Missing cloud file system scheme`) | N/A — use cloud URI |

**This notebook uses:** read Delta from Volume → `saveAsTable`.

---
